# NLP Model Training — Oven Product Matching

## Step 142 — NLP Problem Definition & Modeling Strategy

### Objective
Develop and evaluate machine learning models that extract structured
technical requirements from natural-language customer requests.

### Dataset
- 1,967 provisionally validated customer requests
- 7,727 structured requirements
- 24 product characteristics
- 99 characteristic-value combinations

### Problem Definition
The system must identify multiple technical requirements from a single
customer request.

This is a multi-label text classification problem because one customer
request may refer to several characteristic-value combinations.

### Modeling Strategy

**Phase 1 — Characteristic-Value Prediction**
- Input: Natural-language customer request
- Output: One or more characteristic-value labels
- Baseline: TF-IDF + OneVsRest Logistic Regression

**Phase 2 — Requirement Priority Prediction**
- Distinguish MUST and PREFERRED requirements.
- Evaluate priority prediction separately.

**Phase 3 — Structured Requirement Extraction**
- Combine predicted characteristic-value labels and priorities.
- Assign EQUAL or CONTAINS according to the characteristic definition.

**Phase 4 — Product Matching**
- Match extracted requirements against the Product Master.
- Evaluate mandatory requirements and optional preferences separately.

### Evaluation
- Micro Precision, Recall and F1-score
- Macro Precision, Recall and F1-score
- Exact-match accuracy
- Per-label performance
- Independent linguistic generalization test

### Methodological Considerations
The dataset was generated synthetically and paraphrased using an LLM.
Consequently, performance on a random test split may overestimate
generalization to real customer language.

A separate evaluation set with different linguistic patterns will be
used to investigate this limitation.

In [1]:
# Step 142 - Load NLP training datasets

import pandas as pd
from pathlib import Path

# Project paths
DATA_DIR = Path("../data/processed")

requests_path = DATA_DIR / "nlp_customer_requests.csv"
ground_truth_path = DATA_DIR / "nlp_ground_truth.csv"

# Load datasets
requests_df = pd.read_csv(requests_path)
ground_truth_df = pd.read_csv(ground_truth_path)

print("=== NLP TRAINING DATASETS ===")

print("Customer requests:", requests_df.shape)
print("Ground truth:", ground_truth_df.shape)

print("\n=== CUSTOMER REQUEST COLUMNS ===")
print(requests_df.columns.tolist())

print("\n=== GROUND TRUTH COLUMNS ===")
print(ground_truth_df.columns.tolist())

# Verify request alignment
request_ids = set(requests_df["REQUEST_ID"])
ground_truth_ids = set(ground_truth_df["REQUEST_ID"])

assert request_ids == ground_truth_ids, (   #assert statement checks if the two sets of request IDs are equal. If they are not equal, it raises an AssertionError with the specified message.
    "Request IDs do not match!"
)

assert requests_df["REQUEST_ID"].is_unique, (   #assert statement checks if the "REQUEST_ID" column in the requests_df DataFrame contains unique values. If there are duplicate request IDs, it raises an AssertionError with the specified message.
    "Duplicate customer request IDs detected!"
)

assert requests_df["LLM_TEXT"].notna().all(), ( #assert statement checks if all values in the "LLM_TEXT" column of the requests_df DataFrame are not null (i.e., they contain valid text). If any value is null, it raises an AssertionError with the specified message.
    "Missing customer texts detected!"
)

print("\n=== DATASET VALIDATION ===")
print("Unique requests:", len(request_ids))
print("Total requirements:", len(ground_truth_df))

print("\n=== SAMPLE CUSTOMER REQUEST ===")

sample_id = requests_df.iloc[0]["REQUEST_ID"]

print("Request ID:", sample_id)

print(
    "Customer Text:",
    requests_df.iloc[0]["LLM_TEXT"]
)

print("\nGround Truth:")

display(
    ground_truth_df[
        ground_truth_df["REQUEST_ID"] == sample_id
    ][
        [
            "CHARACTERISTIC",
            "VALUE",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ]
)

print("\nSTEP 142: COMPLETED")

=== NLP TRAINING DATASETS ===
Customer requests: (1967, 4)
Ground truth: (7727, 6)

=== CUSTOMER REQUEST COLUMNS ===
['REQUEST_ID', 'LLM_TEXT', 'INPUT_TOKENS', 'OUTPUT_TOKENS']

=== GROUND TRUTH COLUMNS ===
['REQUEST_ID', 'CHARACTERISTIC', 'VALUE', 'VALUE_DEFINITION', 'OPERATOR', 'REQUIREMENT_TYPE']

=== DATASET VALIDATION ===
Unique requests: 1967
Total requirements: 7727

=== SAMPLE CUSTOMER REQUEST ===
Request ID: R0001
Customer Text: I'm looking for a steam oven that's exactly 45 cm wide.

Ground Truth:


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,OVEN_TYPE,V002,EQUAL,MUST
1,WIDTH_CM,V001,EQUAL,MUST



STEP 142: COMPLETED


## Step 143 — Multi-Label Target Encoding

**Objective:** Convert structured ground truth requirements into a
multi-label target matrix suitable for supervised machine learning.

**Method:**
1. Combine CHARACTERISTIC and VALUE into a unique target label.
2. Represent each customer request as a binary vector.
3. Assign 1 when a characteristic-value pair is requested and 0 otherwise.
4. Align target rows with customer request texts.
5. Verify label coverage and dataset integrity.

**Example:**

A request containing WIDTH_CM=V001 and OVEN_TYPE=V002 will have
a value of 1 for these two labels and 0 for all other labels.

**Expected Outcome:** A binary target matrix with one row per customer
request and one column per characteristic-value combination.

**Important:** MUST/PREFERRED and EQUAL/CONTAINS information will be
preserved in the original ground truth and modeled separately.

In [2]:
# Step 143 - Create multi-label target matrix

import pandas as pd

# Create unique labels
ground_truth_df["TARGET_LABEL"] = (
    ground_truth_df["CHARACTERISTIC"].astype(str)
    + "__"
    + ground_truth_df["VALUE"].astype(str)
)

# Create binary target matrix
target_matrix = pd.crosstab(
    ground_truth_df["REQUEST_ID"],
    ground_truth_df["TARGET_LABEL"]
)

# Convert counts to binary values
target_matrix = (target_matrix > 0).astype(int) 

# Align target matrix with customer request order
target_matrix = target_matrix.reindex(
    requests_df["REQUEST_ID"],
    fill_value=0
)

# Prepare input texts and targets
X = requests_df["LLM_TEXT"].reset_index(drop=True)
y = target_matrix.reset_index(drop=True)

# Validate alignment
assert len(X) == len(y)
assert len(X) == 1967

assert set(y.to_numpy().ravel()) <= {0, 1}

# Check label coverage
label_counts = y.sum(axis=0).sort_values()

print("=== MULTI-LABEL TARGET MATRIX ===")
print("Input texts:", len(X))
print("Target matrix shape:", y.shape)
print("Unique target labels:", y.shape[1])

print("\n=== LABEL DISTRIBUTION ===")
print("Minimum examples per label:", label_counts.min())
print("Maximum examples per label:", label_counts.max())
print("Average examples per label:", round(label_counts.mean(), 2))

print("\n=== RARE LABELS ===")
display(label_counts.head(15).to_frame("REQUEST_COUNT"))

print("\n=== SAMPLE TARGET MATRIX ===")
display(y.iloc[:5, :10])

print("\n=== LABELS FOR FIRST CUSTOMER REQUEST ===")

first_labels = y.columns[y.iloc[0] == 1].tolist()

print("Request ID:", requests_df.iloc[0]["REQUEST_ID"])
print("Customer text:", X.iloc[0])
print("Target labels:", first_labels)

print("\nSTEP 143: COMPLETED")

=== MULTI-LABEL TARGET MATRIX ===
Input texts: 1967
Target matrix shape: (1967, 99)
Unique target labels: 99

=== LABEL DISTRIBUTION ===
Minimum examples per label: 13
Maximum examples per label: 197
Average examples per label: 78.05

=== RARE LABELS ===


,REQUEST_COUNT
TARGET_LABEL,
POWER_CABLE__V002,13
OVEN_FUNCTIONS__Pyrolysis,20
POWER_CABLE__V005,21
KNOB_TYPE__V004,23
POWER_CABLE__V001,23
KNOB_COLOR__V004,29
KNOB_TYPE__V001,30
POWER_CABLE__V004,30
POWER_CABLE__V003,30



=== SAMPLE TARGET MATRIX ===


TARGET_LABEL,BAKING_TRAY__V001,BAKING_TRAY__V002,BAKING_TRAY__V003,CLEANING__V001,CLEANING__V002,CLEANING__V003,CLEANING__V004,ENERGY_CLASS__V001,ENERGY_CLASS__V002,ENERGY_CLASS__V003
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0



=== LABELS FOR FIRST CUSTOMER REQUEST ===
Request ID: R0001
Customer text: I'm looking for a steam oven that's exactly 45 cm wide.
Target labels: ['OVEN_TYPE__V002', 'WIDTH_CM__V001']

STEP 143: COMPLETED


## Step 144 — Train / Validation / Test Split

**Objective:** Divide the validated customer requests into independent training, validation and test datasets.

**Method:** Split the data at REQUEST_ID level using iterative multilabel stratification to preserve the distribution of rare characteristic–value labels as much as possible.

**Dataset split:**
- Training: 70%
- Validation: 15%
- Test: 15%
**Important considerations:**
- Each customer request must belong to only one dataset.
- Ground Truth requirements must remain linked to their original request.
- The test dataset must not be used during model selection or hyperparameter tuning.
- Multilabel stratification is preferred because each request can contain multiple target labels.
- Since all requests are synthetic and many share similar language patterns, a random split alone does not guarantee generalization to genuinely new customer language. We will report this limitation.

In [3]:
# Step 144A - Identify existing dataset variables

import pandas as pd

print("=== AVAILABLE DATAFRAMES ===")

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(
            f"{name}: {value.shape} | "
            f"Columns: {list(value.columns[:6])}"
        )

=== AVAILABLE DATAFRAMES ===
requests_df: (1967, 4) | Columns: ['REQUEST_ID', 'LLM_TEXT', 'INPUT_TOKENS', 'OUTPUT_TOKENS']
ground_truth_df: (7727, 7) | Columns: ['REQUEST_ID', 'CHARACTERISTIC', 'VALUE', 'VALUE_DEFINITION', 'OPERATOR', 'REQUIREMENT_TYPE']
target_matrix: (1967, 99) | Columns: ['BAKING_TRAY__V001', 'BAKING_TRAY__V002', 'BAKING_TRAY__V003', 'CLEANING__V001', 'CLEANING__V002', 'CLEANING__V003']
y: (1967, 99) | Columns: ['BAKING_TRAY__V001', 'BAKING_TRAY__V002', 'BAKING_TRAY__V003', 'CLEANING__V001', 'CLEANING__V002', 'CLEANING__V003']


In [4]:
# Step 144 - Multilabel train / validation / test split

import sys
import subprocess
import numpy as np
import pandas as pd

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "iterative-stratification"
    ])
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

customer_requests = requests_df.copy()

# Use the target matrix prepared in Step 143.
# Ensure the target matrix and customer requests have the same row order.

assert len(customer_requests) == len(target_matrix), (
    "Customer requests and target matrix have different row counts."
)

assert customer_requests["REQUEST_ID"].is_unique, (
    "Duplicate REQUEST_ID values detected."
)

X = customer_requests["LLM_TEXT"].reset_index(drop=True)
# Align target labels with customer request IDs
label_columns = target_matrix.columns

aligned_targets = (
    ground_truth_df
    .assign(
        TARGET_LABEL=lambda df:
            df["CHARACTERISTIC"].astype(str)
            + "__"
            + df["VALUE"].astype(str)
    )
    .assign(PRESENT=1)
    .pivot_table(
        index="REQUEST_ID",
        columns="TARGET_LABEL",
        values="PRESENT",
        aggfunc="max",
        fill_value=0
    )
)

assert set(customer_requests["REQUEST_ID"]) == set(aligned_targets.index)
assert set(label_columns) == set(aligned_targets.columns)

y = (
    aligned_targets
    .reindex(customer_requests["REQUEST_ID"])
    .loc[:, label_columns]
    .reset_index(drop=True)
    .astype(int)
)

assert y.shape == (1967, 99)

request_ids = (
    customer_requests["REQUEST_ID"]
    .reset_index(drop=True)
)

assert len(X) == len(y) == len(request_ids)


# First split: 70% training, 30% temporary

splitter_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    splitter_1.split(X, y)
)


# Second split: temporary data into validation and test

splitter_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_relative_idx, test_relative_idx = next(
    splitter_2.split(
        X.iloc[temp_idx],
        y.iloc[temp_idx]
    )
)

val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]


# Create datasets

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]
y_test = y.iloc[test_idx]


# Validate request separation

train_ids = set(request_ids.iloc[train_idx])
val_ids = set(request_ids.iloc[val_idx])
test_ids = set(request_ids.iloc[test_idx])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

assert len(train_ids | val_ids | test_ids) == len(request_ids)


# Check label coverage

label_coverage = pd.DataFrame({
    "TRAIN": y_train.sum(),
    "VALIDATION": y_val.sum(),
    "TEST": y_test.sum()
})

print("=== DATASET SPLIT ===")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print("Total:", len(X_train) + len(X_val) + len(X_test))

print("\n=== LABEL COVERAGE ===")
print("Total labels:", y.shape[1])
print("Labels missing from training:",
      (label_coverage["TRAIN"] == 0).sum())
print("Labels missing from validation:",
      (label_coverage["VALIDATION"] == 0).sum())
print("Labels missing from test:",
      (label_coverage["TEST"] == 0).sum())

print("\n=== LOWEST TRAINING LABEL COUNTS ===")
display(
    label_coverage
    .sort_values("TRAIN")
    .head(15)
)

print("\nSTEP 144: COMPLETED")

=== DATASET SPLIT ===
Training: 1372
Validation: 297
Test: 298
Total: 1967

=== LABEL COVERAGE ===
Total labels: 99
Labels missing from training: 0
Labels missing from validation: 0
Labels missing from test: 0

=== LOWEST TRAINING LABEL COUNTS ===


,TRAIN,VALIDATION,TEST
TARGET_LABEL,,,
POWER_CABLE__V002,9,2,2
OVEN_FUNCTIONS__Pyrolysis,14,3,3
POWER_CABLE__V005,15,3,3
POWER_CABLE__V001,16,3,4
KNOB_TYPE__V004,16,3,4
KNOB_COLOR__V004,20,4,5
POWER_CABLE__V003,21,4,5
POWER_CABLE__V004,21,4,5
KNOB_TYPE__V001,21,4,5



STEP 144: COMPLETED


## Step 145 — Baseline NLP Model: TF-IDF + Logistic Regression

**Objective:** Train the first supervised machine learning model to extract technical product characteristics and values from natural-language customer requests.

**Method:**
1. Convert customer texts into numerical features using TF-IDF.
2. Train a Logistic Regression classifier for each characteristic-value label.
3. Use a One-vs-Rest approach for multilabel classification.
4. Evaluate predictions on the validation dataset.

**Model architecture:**

Customer Request → TF-IDF Vectorization → Logistic Regression → Multilabel Predictions

**Training configuration:**
- Vectorization: TF-IDF with unigrams and bigrams.
- Classification: Logistic Regression.
- Multilabel strategy: One-vs-Rest.
- Initial prediction threshold: 0.50.
- Training dataset: 1,372 requests.
- Validation dataset: 297 requests.

**Evaluation metrics:**
- Micro Precision
- Micro Recall
- Micro F1-score
- Macro F1-score
- Exact Match Accuracy

**Important:** The test dataset is reserved for final evaluation and will not be used during model development.

In [5]:
# Step 145 - Train baseline NLP model

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# --------------------------------------------------
# 1. TF-IDF VECTORIZATION
# --------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit only on training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform validation data using the same vocabulary
X_val_tfidf = tfidf_vectorizer.transform(X_val)

print("=== TF-IDF VECTORIZATION ===")
print("Training matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_val_tfidf.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))


# --------------------------------------------------
# 2. LOGISTIC REGRESSION MODEL
# --------------------------------------------------

base_classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

nlp_model = OneVsRestClassifier(
    base_classifier,
    n_jobs=-1
)

# Train the model
nlp_model.fit(X_train_tfidf, y_train)

print("\n=== MODEL TRAINING ===")
print("Model: TF-IDF + Logistic Regression")
print("Classification strategy: One-vs-Rest")
print("Number of labels:", y_train.shape[1])
print("Training completed successfully.")


# --------------------------------------------------
# 3. VALIDATION PREDICTIONS
# --------------------------------------------------

# Predict probabilities for all labels
y_val_proba = nlp_model.predict_proba(X_val_tfidf)

# Convert probabilities into binary predictions
threshold = 0.50

y_val_pred = (
    y_val_proba >= threshold
).astype(int)


# --------------------------------------------------
# 4. PERFORMANCE EVALUATION
# --------------------------------------------------

micro_precision = precision_score(
    y_val,
    y_val_pred,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_val,
    y_val_pred,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro",
    zero_division=0
)

exact_match_accuracy = accuracy_score(
    y_val,
    y_val_pred
)


print("\n=== BASELINE VALIDATION RESULTS ===")

print(f"Micro Precision: {micro_precision:.4f}")
print(f"Micro Recall:    {micro_recall:.4f}")
print(f"Micro F1-score:  {micro_f1:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")
print(f"Exact Match:     {exact_match_accuracy:.4f}")

print("\nSTEP 145: COMPLETED")

=== TF-IDF VECTORIZATION ===
Training matrix: (1372, 2076)
Validation matrix: (297, 2076)
Vocabulary size: 2076

=== MODEL TRAINING ===
Model: TF-IDF + Logistic Regression
Classification strategy: One-vs-Rest
Number of labels: 99
Training completed successfully.

=== BASELINE VALIDATION RESULTS ===
Micro Precision: 0.8251
Micro Recall:    0.9308
Micro F1-score:  0.8748
Macro F1-score:  0.8855
Exact Match:     0.3906

STEP 145: COMPLETED


In [6]:
# Step 146 - Label-level performance analysis

import pandas as pd
import numpy as np

from sklearn.metrics import precision_recall_fscore_support

# Calculate metrics for each target label
precision, recall, f1, support = precision_recall_fscore_support(
    y_val,
    y_val_pred,
    average=None,
    zero_division=0
)

# Create performance report
label_performance = pd.DataFrame({
    "TARGET_LABEL": y_val.columns,
    "PRECISION": precision,
    "RECALL": recall,
    "F1_SCORE": f1,
    "SUPPORT": support
})

# Sort by lowest F1-score
label_performance = label_performance.sort_values(
    ["F1_SCORE", "SUPPORT"],
    ascending=[True, True]
).reset_index(drop=True)

print("=== LABEL-LEVEL PERFORMANCE ===")
print("Total labels:", len(label_performance))

print("\n=== LOWEST PERFORMING LABELS ===")
display(label_performance.head(20))

print("\n=== HIGHEST PERFORMING LABELS ===")
display(
    label_performance.sort_values(
        "F1_SCORE",
        ascending=False
    ).head(10)
)

print("\n=== RARE LABEL PERFORMANCE ===")

rare_labels = label_performance[
    label_performance["SUPPORT"] <= 5
]

print("Labels with 5 or fewer validation examples:", len(rare_labels))

display(rare_labels)

print("\n=== PERFORMANCE SUMMARY ===")

print(
    "Labels with F1 = 1.0:",
    (label_performance["F1_SCORE"] == 1.0).sum()
)

print(
    "Labels with F1 < 0.5:",
    (label_performance["F1_SCORE"] < 0.5).sum()
)

print(
    "Labels with F1 = 0:",
    (label_performance["F1_SCORE"] == 0).sum()
)

print("\nSTEP 146: COMPLETED")

=== LABEL-LEVEL PERFORMANCE ===
Total labels: 99

=== LOWEST PERFORMING LABELS ===


,TARGET_LABEL,PRECISION,RECALL,F1_SCORE,SUPPORT
0,ENERGY_CLASS__V002,0.400000,0.727273,0.516129,11
1,HANDLE_COLOR__V002,0.500000,0.666667,0.571429,6
2,ENERGY_CLASS__V003,0.428571,0.857143,0.571429,7
3,CLEANING__V004,0.538462,0.700000,0.608696,10
4,ENERGY_CLASS__V001,0.550000,0.687500,0.611111,16
5,OVEN_FUNCTIONS__Grill,0.566667,0.708333,0.629630,24
6,PANEL_COLOR__V003,0.600000,0.666667,0.631579,9
7,OVEN_FUNCTIONS__DoubleGrill+Turnspit,0.611111,0.846154,0.709677,13
8,HANDLE_COLOR__V004,0.625000,0.833333,0.714286,6
9,OVEN_FUNCTIONS__Bottom,0.625000,0.833333,0.714286,24



=== HIGHEST PERFORMING LABELS ===


,TARGET_LABEL,PRECISION,RECALL,F1_SCORE,SUPPORT
98,WIDTH_CM__V002,1.0,1.0,1.0,23
80,ROASTING_TRAY__V003,1.0,1.0,1.0,10
78,NUMBER_OF_FUNCTIONS__V002,1.0,1.0,1.0,9
77,HANDLE_COLOR__V001,1.0,1.0,1.0,9
76,WIDTH_CM__V003,1.0,1.0,1.0,8
74,HANDLE_TYPE__V002,1.0,1.0,1.0,7
73,HANDLE_TYPE__V001,1.0,1.0,1.0,7
72,NUMBER_OF_FUNCTIONS__V001,1.0,1.0,1.0,6
71,KNOB_TYPE__V005,1.0,1.0,1.0,6
70,OVEN_VOLUME_L__V004,1.0,1.0,1.0,5



=== RARE LABEL PERFORMANCE ===
Labels with 5 or fewer validation examples: 14


,TARGET_LABEL,PRECISION,RECALL,F1_SCORE,SUPPORT
12,KNOB_COLOR__V004,0.571429,1.00,0.727273,4
15,OVEN_FUNCTIONS__Pyrolysis,0.600000,1.00,0.750000,3
16,OVEN_VOLUME_L__V001,1.000000,0.60,0.750000,5
33,POWER_CABLE__V004,1.000000,0.75,0.857143,4
40,HANDLE_TYPE__V005,1.000000,0.80,0.888889,5
41,KNOB_COLOR__V002,1.000000,0.80,0.888889,5
63,POWER_CABLE__V002,1.000000,1.00,1.000000,2
64,KNOB_TYPE__V004,1.000000,1.00,1.000000,3
65,POWER_CABLE__V001,1.000000,1.00,1.000000,3
66,POWER_CABLE__V005,1.000000,1.00,1.000000,3



=== PERFORMANCE SUMMARY ===
Labels with F1 = 1.0: 36
Labels with F1 < 0.5: 0
Labels with F1 = 0: 0

STEP 146: COMPLETED


## Step 147 — Error Analysis: False Positives and False Negatives

**Objective:** Identify which characteristic-value labels are most frequently
predicted incorrectly by the baseline NLP model.

**Method:**
- Compare validation ground truth with model predictions.
- Calculate False Positives (FP) and False Negatives (FN) for each label.
- Identify labels requiring further investigation.
- Examine customer texts associated with incorrect predictions.

**Expected Outcome:**
Understand the main error patterns before training and comparing
alternative NLP models.

**Note:** The analysis uses only the validation dataset.
The test dataset remains untouched for final evaluation.

In [7]:
# Step 147 - False Positive and False Negative Analysis

import numpy as np
import pandas as pd

# Convert validation labels and predictions to arrays
y_true_array = np.asarray(y_val)
y_pred_array = np.asarray(y_val_pred)

# Validate dimensions
assert y_true_array.shape == y_pred_array.shape, (
    "Ground truth and prediction shapes do not match."
)

# Calculate errors for each label
false_positives = (
    (y_true_array == 0) &
    (y_pred_array == 1)
).sum(axis=0)

false_negatives = (
    (y_true_array == 1) &
    (y_pred_array == 0)
).sum(axis=0)

true_positives = (
    (y_true_array == 1) &
    (y_pred_array == 1)
).sum(axis=0)

# Build error analysis dataframe
error_analysis = pd.DataFrame({
    "TARGET_LABEL": y_val.columns,
    "TP": true_positives,
    "FP": false_positives,
    "FN": false_negatives
})

error_analysis["TOTAL_ERRORS"] = (
    error_analysis["FP"] +
    error_analysis["FN"]
)

# Sort by total errors
error_analysis = error_analysis.sort_values(
    "TOTAL_ERRORS",
    ascending=False
).reset_index(drop=True)

print("=== ERROR ANALYSIS SUMMARY ===")

print("Total False Positives:", error_analysis["FP"].sum())
print("Total False Negatives:", error_analysis["FN"].sum())

print("\n=== TOP 20 LABELS WITH MOST ERRORS ===")

display(error_analysis.head(20))

print("\n=== LABELS WITH MOST FALSE POSITIVES ===")

display(
    error_analysis.sort_values(
        "FP",
        ascending=False
    ).head(10)
)

print("\n=== LABELS WITH MOST FALSE NEGATIVES ===")

display(
    error_analysis.sort_values(
        "FN",
        ascending=False
    ).head(10)
)

print("\n=== ERROR-FREE LABELS ===")

print(
    "Labels without errors:",
    (error_analysis["TOTAL_ERRORS"] == 0).sum()
)

print("\nSTEP 147: COMPLETED")

=== ERROR ANALYSIS SUMMARY ===
Total False Positives: 231
Total False Negatives: 81

=== TOP 20 LABELS WITH MOST ERRORS ===


,TARGET_LABEL,TP,FP,FN,TOTAL_ERRORS
0,OVEN_FUNCTIONS__Grill,17,13,7,20
1,OVEN_FUNCTIONS__Bottom,20,12,4,16
2,OVEN_FUNCTIONS__DoubleGrill+Fan,20,15,0,15
3,ENERGY_CLASS__V002,8,12,3,15
4,ENERGY_CLASS__V001,11,9,5,14
5,OVEN_FUNCTIONS__DoubleGrill,19,8,3,11
6,OVEN_FUNCTIONS__Bottom+Fan,31,9,1,10
7,OVEN_FUNCTIONS__DoubleGrill+Turnspit,11,7,2,9
8,OVEN_FUNCTIONS__Top+Bottom,32,9,0,9
9,OVEN_FUNCTIONS__Grill+Turnspit,12,7,2,9



=== LABELS WITH MOST FALSE POSITIVES ===


,TARGET_LABEL,TP,FP,FN,TOTAL_ERRORS
2,OVEN_FUNCTIONS__DoubleGrill+Fan,20,15,0,15
0,OVEN_FUNCTIONS__Grill,17,13,7,20
3,ENERGY_CLASS__V002,8,12,3,15
1,OVEN_FUNCTIONS__Bottom,20,12,4,16
11,WIFI__V002,15,9,0,9
4,ENERGY_CLASS__V001,11,9,5,14
6,OVEN_FUNCTIONS__Bottom+Fan,31,9,1,10
8,OVEN_FUNCTIONS__Top+Bottom,32,9,0,9
5,OVEN_FUNCTIONS__DoubleGrill,19,8,3,11
12,ENERGY_CLASS__V003,6,8,1,9



=== LABELS WITH MOST FALSE NEGATIVES ===


,TARGET_LABEL,TP,FP,FN,TOTAL_ERRORS
0,OVEN_FUNCTIONS__Grill,17,13,7,20
4,ENERGY_CLASS__V001,11,9,5,14
1,OVEN_FUNCTIONS__Bottom,20,12,4,16
3,ENERGY_CLASS__V002,8,12,3,15
5,OVEN_FUNCTIONS__DoubleGrill,19,8,3,11
10,CLEANING__V004,7,6,3,9
14,PANEL_COLOR__V003,6,4,3,7
19,OVEN_FUNCTIONS__Grill+Fan,11,3,3,6
17,HANDLE_COLOR__V002,4,4,2,6
47,NUMBER_OF_FUNCTIONS__V004,9,0,2,2



=== ERROR-FREE LABELS ===
Labels without errors: 36

STEP 147: COMPLETED


## Step 148 — Linear SVM Model Training

**Objective:** Train a second supervised learning model and compare its
performance with the Logistic Regression baseline.

**Method:**
- Reuse the TF-IDF features generated from the training dataset.
- Train a Linear Support Vector Machine (LinearSVC).
- Apply the One-vs-Rest strategy for multi-label classification.
- Evaluate the model using the same validation dataset.

**Evaluation Metrics:**
- Micro Precision
- Micro Recall
- Micro F1-score
- Macro F1-score
- Exact Match Accuracy

**Expected Outcome:**
Determine whether Linear SVM provides better classification performance
than the Logistic Regression baseline.

**Note:** The test dataset is reserved for final evaluation.

In [8]:
# Step 148 - Linear SVM Model Training

import numpy as np

from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Initialize Linear SVM model
svm_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        class_weight=None,
        max_iter=5000,
        random_state=42
    )
)

# Train using the existing TF-IDF training matrix
svm_model.fit(X_train_tfidf, y_train)

# Predict validation labels
y_val_pred_svm = svm_model.predict(X_val_tfidf)

# Calculate validation metrics
svm_precision = precision_score(
    y_val, y_val_pred_svm,
    average="micro",
    zero_division=0
)

svm_recall = recall_score(
    y_val, y_val_pred_svm,
    average="micro",
    zero_division=0
)

svm_micro_f1 = f1_score(
    y_val, y_val_pred_svm,
    average="micro",
    zero_division=0
)

svm_macro_f1 = f1_score(
    y_val, y_val_pred_svm,
    average="macro",
    zero_division=0
)

svm_exact_match = accuracy_score(
    y_val, y_val_pred_svm
)

print("=== LINEAR SVM TRAINING ===")
print("Model: TF-IDF + Linear SVM")
print("Classification strategy: One-vs-Rest")
print("Number of labels:", y_train.shape[1])

print("\n=== LINEAR SVM VALIDATION RESULTS ===")
print(f"Micro Precision: {svm_precision:.4f}")
print(f"Micro Recall:    {svm_recall:.4f}")
print(f"Micro F1-score:  {svm_micro_f1:.4f}")
print(f"Macro F1-score:  {svm_macro_f1:.4f}")
print(f"Exact Match:     {svm_exact_match:.4f}")

print("\nSTEP 148: COMPLETED")

=== LINEAR SVM TRAINING ===
Model: TF-IDF + Linear SVM
Classification strategy: One-vs-Rest
Number of labels: 99

=== LINEAR SVM VALIDATION RESULTS ===
Micro Precision: 0.9705
Micro Recall:    0.8138
Micro F1-score:  0.8853
Macro F1-score:  0.8490
Exact Match:     0.5185

STEP 148: COMPLETED


In [9]:
# Step 149 - Linear SVM Decision Threshold Optimization

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Identify the trained SVM model and validation dataset
print("=== AVAILABLE MODEL VARIABLES ===")

for name in [
    "svm_model",
    "svm_classifier",
    "linear_svm",
    "X_val_tfidf",
    "X_val_vec",
    "y_val"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

# Set the variable names used in Step 148
# Update these two lines if your names differ.
svm_estimator = svm_model
X_validation = X_val_tfidf

# Calculate decision scores for each label
decision_scores = svm_estimator.decision_function(
    X_validation
)

# Convert ground truth to NumPy array
y_true = np.asarray(y_val)

assert decision_scores.shape == y_true.shape, (
    "Decision scores and ground truth shapes do not match."
)

# Test different decision thresholds
thresholds = np.arange(-0.6, 0.61, 0.1)

results = []

for threshold in thresholds:

    y_pred = (
        decision_scores >= threshold
    ).astype(int)

    results.append({
        "THRESHOLD": round(float(threshold), 2),

        "MICRO_PRECISION": precision_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),

        "MICRO_RECALL": recall_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),

        "MICRO_F1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),

        "MACRO_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),

        "EXACT_MATCH": accuracy_score(
            y_true, y_pred
        )
    })

threshold_results = pd.DataFrame(results)

print("=== LINEAR SVM THRESHOLD COMPARISON ===")

display(
    threshold_results.round(4)
)

print("\n=== BEST THRESHOLD BY MICRO F1 ===")

best_f1_row = threshold_results.loc[
    threshold_results["MICRO_F1"].idxmax()
]

display(
    best_f1_row.to_frame().T.round(4)
)

print("\n=== BEST THRESHOLD BY EXACT MATCH ===")

best_exact_row = threshold_results.loc[
    threshold_results["EXACT_MATCH"].idxmax()
]

display(
    best_exact_row.to_frame().T.round(4)
)

print("\nSTEP 149: COMPLETED")

=== AVAILABLE MODEL VARIABLES ===
svm_model: OneVsRestClassifier
X_val_tfidf: csr_matrix
y_val: DataFrame
=== LINEAR SVM THRESHOLD COMPARISON ===


,THRESHOLD,MICRO_PRECISION,MICRO_RECALL,MICRO_F1,MACRO_F1,EXACT_MATCH
0,-0.6,0.7874,0.9744,0.8710,0.8917,0.3300
1,-0.5,0.8471,0.9607,0.9004,0.9103,0.4747
2,-0.4,0.8935,0.9453,0.9187,0.9224,0.5455
3,-0.3,0.9304,0.9240,0.9272,0.9250,0.6397
4,-0.2,0.9444,0.8992,0.9213,0.9079,0.6229
5,-0.1,0.9598,0.8557,0.9047,0.8836,0.5758
6,-0.0,0.9705,0.8138,0.8853,0.8490,0.5185
7,0.1,0.9761,0.7686,0.8600,0.8217,0.4411
8,0.2,0.9807,0.6943,0.8130,0.7511,0.3603
9,0.3,0.9839,0.6251,0.7645,0.6988,0.3165



=== BEST THRESHOLD BY MICRO F1 ===


,THRESHOLD,MICRO_PRECISION,MICRO_RECALL,MICRO_F1,MACRO_F1,EXACT_MATCH
3,-0.3,0.9304,0.924,0.9272,0.925,0.6397



=== BEST THRESHOLD BY EXACT MATCH ===


,THRESHOLD,MICRO_PRECISION,MICRO_RECALL,MICRO_F1,MACRO_F1,EXACT_MATCH
3,-0.3,0.9304,0.924,0.9272,0.925,0.6397



STEP 149: COMPLETED


In [10]:
# Step 150 - Logistic Regression Decision Threshold Optimization

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

for name, obj in globals().items():
    if hasattr(obj, "estimators_"):
        print(name, ":", type(obj).__name__)

print("=== AVAILABLE LOGISTIC REGRESSION VARIABLES ===")

for name in [
    "lr_model",
    "logistic_model",
    "logreg_model",
    "logistic_regression_model",
    "X_val_tfidf",
    "y_val"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

# Identify the Logistic Regression model from Step 145.
# Change this line only if the model has a different name.
lr_estimator = nlp_model

# Validation data
X_validation = X_val_tfidf
y_true = np.asarray(y_val)

# Predict probabilities for each label.
# Each estimator in OneVsRestClassifier predicts one label.
probabilities = np.column_stack([
    estimator.predict_proba(X_validation)[:, 1]
    for estimator in lr_estimator.estimators_
])

assert probabilities.shape == y_true.shape, (
    "Probability matrix and ground truth shapes do not match."
)

thresholds = np.arange(0.2, 0.81, 0.05)

results = []

for threshold in thresholds:

    y_pred = (
        probabilities >= threshold
    ).astype(int)

    results.append({
        "THRESHOLD": round(float(threshold), 2),

        "MICRO_PRECISION": precision_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),

        "MICRO_RECALL": recall_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),

        "MICRO_F1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),

        "MACRO_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),

        "EXACT_MATCH": accuracy_score(
            y_true, y_pred
        )
    })

lr_threshold_results = pd.DataFrame(results)

print("\n=== LOGISTIC REGRESSION THRESHOLD COMPARISON ===")

display(
    lr_threshold_results.round(4)
)

print("\n=== BEST THRESHOLD BY MICRO F1 ===")

best_lr_f1 = lr_threshold_results.loc[
    lr_threshold_results["MICRO_F1"].idxmax()
]

display(
    best_lr_f1.to_frame().T.round(4)
)

print("\n=== BEST THRESHOLD BY EXACT MATCH ===")

best_lr_exact = lr_threshold_results.loc[
    lr_threshold_results["EXACT_MATCH"].idxmax()
]

display(
    best_lr_exact.to_frame().T.round(4)
)

print("\nSTEP 150: COMPLETED")

nlp_model : OneVsRestClassifier
svm_model : OneVsRestClassifier
svm_estimator : OneVsRestClassifier
=== AVAILABLE LOGISTIC REGRESSION VARIABLES ===
X_val_tfidf: csr_matrix
y_val: DataFrame

=== LOGISTIC REGRESSION THRESHOLD COMPARISON ===


,THRESHOLD,MICRO_PRECISION,MICRO_RECALL,MICRO_F1,MACRO_F1,EXACT_MATCH
0,0.20,0.3746,0.9949,0.5443,0.6016,0.0034
1,0.25,0.4842,0.9923,0.6508,0.7023,0.0067
2,0.30,0.5842,0.9863,0.7338,0.7719,0.0774
3,0.35,0.6657,0.9778,0.7921,0.8214,0.1481
4,0.40,0.7248,0.9693,0.8294,0.8500,0.2222
5,0.45,0.7793,0.9496,0.8560,0.8663,0.3131
6,0.50,0.8251,0.9308,0.8748,0.8855,0.3906
7,0.55,0.8543,0.9009,0.8770,0.8784,0.4175
8,0.60,0.8784,0.8634,0.8708,0.8640,0.4175
9,0.65,0.9015,0.8130,0.8550,0.8421,0.4108



=== BEST THRESHOLD BY MICRO F1 ===


,THRESHOLD,MICRO_PRECISION,MICRO_RECALL,MICRO_F1,MACRO_F1,EXACT_MATCH
7,0.55,0.8543,0.9009,0.877,0.8784,0.4175



=== BEST THRESHOLD BY EXACT MATCH ===


,THRESHOLD,MICRO_PRECISION,MICRO_RECALL,MICRO_F1,MACRO_F1,EXACT_MATCH
7,0.55,0.8543,0.9009,0.877,0.8784,0.4175



STEP 150: COMPLETED


In [11]:
# Step 151 - Final Linear SVM Test Evaluation

import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Selected model and threshold
final_model = svm_model
final_threshold = -0.30

# Transform test texts using the fitted TF-IDF vectorizer
# Replace 'tfidf_vectorizer' if your vectorizer has a different name.
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Calculate decision scores
test_scores = final_model.decision_function(X_test_tfidf)

# Apply optimized threshold
y_test_true = np.asarray(y_test)

y_test_pred = (
    test_scores >= final_threshold
).astype(int)

assert y_test_true.shape == y_test_pred.shape

# Calculate metrics
test_precision = precision_score(
    y_test_true, y_test_pred,
    average="micro",
    zero_division=0
)

test_recall = recall_score(
    y_test_true, y_test_pred,
    average="micro",
    zero_division=0
)

test_f1_micro = f1_score(
    y_test_true, y_test_pred,
    average="micro",
    zero_division=0
)

test_f1_macro = f1_score(
    y_test_true, y_test_pred,
    average="macro",
    zero_division=0
)

test_exact_match = accuracy_score(
    y_test_true, y_test_pred
)

print("=== FINAL LINEAR SVM TEST RESULTS ===")
print("Model: TF-IDF + Linear SVM")
print("Decision threshold:", final_threshold)
print("Test requests:", len(y_test_true))

print("\n=== PERFORMANCE METRICS ===")
print(f"Micro Precision: {test_precision:.4f}")
print(f"Micro Recall:    {test_recall:.4f}")
print(f"Micro F1-score:  {test_f1_micro:.4f}")
print(f"Macro F1-score:  {test_f1_macro:.4f}")
print(f"Exact Match:     {test_exact_match:.4f}")

print("\nSTEP 151: COMPLETED")

=== FINAL LINEAR SVM TEST RESULTS ===
Model: TF-IDF + Linear SVM
Decision threshold: -0.3
Test requests: 298

=== PERFORMANCE METRICS ===
Micro Precision: 0.9416
Micro Recall:    0.9186
Micro F1-score:  0.9299
Macro F1-score:  0.9130
Exact Match:     0.6477

STEP 151: COMPLETED


In [12]:
# Step 152 - Prepare Requirement Priority Classification Dataset

import pandas as pd

# Use the validated datasets from Step 142
priority_requests = requests_df.copy()
priority_ground_truth = ground_truth_df.copy()

# Select the required columns
priority_ground_truth = priority_ground_truth[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "VALUE_DEFINITION",
        "REQUIREMENT_TYPE"
    ]
].copy()

# Add customer text to each requirement
priority_dataset = priority_ground_truth.merge(
    priority_requests[
        ["REQUEST_ID", "LLM_TEXT"]
    ],
    on="REQUEST_ID",
    how="left",
    validate="many_to_one"
)

# Create a readable requirement description
priority_dataset["REQUIREMENT_DESCRIPTION"] = (
    priority_dataset["CHARACTERISTIC"].astype(str)
    + " = "
    + priority_dataset["VALUE_DEFINITION"].astype(str)
)

# Encode target:
# MUST = 1
# PREFERRED = 0
priority_dataset["PRIORITY_TARGET"] = (
    priority_dataset["REQUIREMENT_TYPE"]
    .map({
        "MUST": 1,
        "PREFERRED": 0
    })
)

# Data integrity checks
assert priority_dataset["LLM_TEXT"].notna().all()
assert priority_dataset["PRIORITY_TARGET"].notna().all()
assert len(priority_dataset) == len(priority_ground_truth)

print("=== PRIORITY CLASSIFICATION DATASET ===")
print("Total requirements:", len(priority_dataset))
print(
    "Unique customer requests:",
    priority_dataset["REQUEST_ID"].nunique()
)

print("\n=== TARGET DISTRIBUTION ===")
print(
    priority_dataset["REQUIREMENT_TYPE"].value_counts()
)

print("\n=== SAMPLE TRAINING RECORDS ===")

display(
    priority_dataset[
        [
            "REQUEST_ID",
            "LLM_TEXT",
            "REQUIREMENT_DESCRIPTION",
            "REQUIREMENT_TYPE",
            "PRIORITY_TARGET"
        ]
    ].head(10)
)

print("\nSTEP 152: COMPLETED")

=== PRIORITY CLASSIFICATION DATASET ===
Total requirements: 7727
Unique customer requests: 1967

=== TARGET DISTRIBUTION ===
REQUIREMENT_TYPE
MUST         5834
PREFERRED    1893
Name: count, dtype: int64

=== SAMPLE TRAINING RECORDS ===


,REQUEST_ID,LLM_TEXT,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PRIORITY_TARGET
0,R0001,I'm looking for a steam oven that's exactly 45...,OVEN_TYPE = STEAM,MUST,1
1,R0001,I'm looking for a steam oven that's exactly 45...,WIDTH_CM = 45,MUST,1
2,R0002,I'm looking for an oven with a white control p...,OVEN_VOLUME_L = 50,MUST,1
3,R0002,I'm looking for an oven with a white control p...,PACKAGING_TYPE = SHRINK,MUST,1
4,R0002,I'm looking for an oven with a white control p...,PANEL_COLOR = WHITE,MUST,1
5,R0003,I'm looking for a built-in combination oven ab...,BAKING_TRAY = 2,MUST,1
6,R0003,I'm looking for a built-in combination oven ab...,KNOB_TYPE = K2,MUST,1
7,R0003,I'm looking for a built-in combination oven ab...,OVEN_TYPE = COMBINATION,MUST,1
8,R0003,I'm looking for a built-in combination oven ab...,PANEL_COLOR = SILVER,PREFERRED,0
9,R0003,I'm looking for a built-in combination oven ab...,PANEL_TYPE = GLASS,MUST,1



STEP 152: COMPLETED


In [13]:
# Step 153 - Priority Dataset Train / Validation / Test Split

import pandas as pd

# Original request IDs from Step 144
# X_train, X_val and X_test contain customer texts.
# We use their indices to retrieve the corresponding REQUEST_IDs.

train_request_ids = set(
    requests_df.iloc[X_train.index]["REQUEST_ID"]
)

val_request_ids = set(
    requests_df.iloc[X_val.index]["REQUEST_ID"]
)

test_request_ids = set(
    requests_df.iloc[X_test.index]["REQUEST_ID"]
)

# Ensure no request appears in multiple groups
assert train_request_ids.isdisjoint(val_request_ids)
assert train_request_ids.isdisjoint(test_request_ids)
assert val_request_ids.isdisjoint(test_request_ids)

# Split priority records by REQUEST_ID
priority_train = priority_dataset[
    priority_dataset["REQUEST_ID"].isin(train_request_ids)
].copy()

priority_val = priority_dataset[
    priority_dataset["REQUEST_ID"].isin(val_request_ids)
].copy()

priority_test = priority_dataset[
    priority_dataset["REQUEST_ID"].isin(test_request_ids)
].copy()

# Integrity checks
assert (
    len(priority_train)
    + len(priority_val)
    + len(priority_test)
    == len(priority_dataset)
)

print("=== PRIORITY DATASET SPLIT ===")

for name, df in [
    ("Training", priority_train),
    ("Validation", priority_val),
    ("Test", priority_test)
]:
    print(
        f"{name}: "
        f"{df['REQUEST_ID'].nunique()} requests | "
        f"{len(df)} requirements"
    )

print("\n=== CLASS DISTRIBUTION ===")

for name, df in [
    ("Training", priority_train),
    ("Validation", priority_val),
    ("Test", priority_test)
]:
    print(f"\n{name}:")
    print(
        df["REQUIREMENT_TYPE"].value_counts(
            normalize=True
        ).round(3)
    )

print("\nSTEP 153: COMPLETED")

=== PRIORITY DATASET SPLIT ===
Training: 1372 requests | 5328 requirements
Validation: 297 requests | 1171 requirements
Test: 298 requests | 1228 requirements

=== CLASS DISTRIBUTION ===

Training:
REQUIREMENT_TYPE
MUST         0.758
PREFERRED    0.242
Name: proportion, dtype: float64

Validation:
REQUIREMENT_TYPE
MUST         0.769
PREFERRED    0.231
Name: proportion, dtype: float64

Test:
REQUIREMENT_TYPE
MUST         0.73
PREFERRED    0.27
Name: proportion, dtype: float64

STEP 153: COMPLETED


In [14]:
# Step 153A - Verify Cross-Model Dataset Isolation

# Request IDs in each priority dataset
train_ids = set(priority_train["REQUEST_ID"])
val_ids = set(priority_val["REQUEST_ID"])
test_ids = set(priority_test["REQUEST_ID"])

# Check for overlapping requests
train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("=== DATA LEAKAGE CHECK ===")

print("Train / Validation overlap:", len(train_val_overlap))
print("Train / Test overlap:", len(train_test_overlap))
print("Validation / Test overlap:", len(val_test_overlap))

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

# Verify all validated requests are represented
all_split_ids = train_ids | val_ids | test_ids
all_request_ids = set(requests_df["REQUEST_ID"])

assert all_split_ids == all_request_ids

print("\n=== DATASET COVERAGE ===")
print("Original requests:", len(all_request_ids))
print("Requests in splits:", len(all_split_ids))
print("Missing requests:", len(all_request_ids - all_split_ids))

print("\nSTEP 153A: COMPLETED")

=== DATA LEAKAGE CHECK ===
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

=== DATASET COVERAGE ===
Original requests: 1967
Requests in splits: 1967
Missing requests: 0

STEP 153A: COMPLETED


In [15]:
# Step 154 - Priority classification baseline
# Model: TF-IDF + Logistic Regression

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# 1. Prepare model inputs
# --------------------------------------------------

def prepare_priority_input(df):

    customer_text = df["LLM_TEXT"].fillna("").astype(str)

    requirement = (
        df["REQUIREMENT_DESCRIPTION"]
        .fillna("")
        .astype(str)
    )

    # Combine customer context with the specific requirement.
    return (
        "CUSTOMER REQUEST: " + customer_text
        + " REQUIREMENT: " + requirement
    )


X_priority_train = prepare_priority_input(priority_train)
X_priority_val = prepare_priority_input(priority_val)

y_priority_train = priority_train["PRIORITY_TARGET"].astype(int)
y_priority_val = priority_val["PRIORITY_TARGET"].astype(int)


# --------------------------------------------------
# 2. Build model pipeline
# --------------------------------------------------

priority_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=10000,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])


# --------------------------------------------------
# 3. Train the model
# --------------------------------------------------

priority_model.fit(
    X_priority_train,
    y_priority_train
)

print("=== PRIORITY MODEL TRAINING ===")
print("Model: TF-IDF + Logistic Regression")
print("Training requirements:", len(X_priority_train))
print("Validation requirements:", len(X_priority_val))
print("Training completed successfully.")


# --------------------------------------------------
# 4. Validation predictions
# --------------------------------------------------

priority_val_pred = priority_model.predict(X_priority_val)


# --------------------------------------------------
# 5. Evaluate performance
# --------------------------------------------------

print("\n=== PRIORITY VALIDATION RESULTS ===")

print(
    "Accuracy:",
    round(accuracy_score(y_priority_val, priority_val_pred), 4)
)

print(
    "Macro Precision:",
    round(
        precision_score(
            y_priority_val,
            priority_val_pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print(
    "Macro Recall:",
    round(
        recall_score(
            y_priority_val,
            priority_val_pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_priority_val,
            priority_val_pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)


# --------------------------------------------------
# 6. Detailed classification report
# --------------------------------------------------

print("\n=== CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_priority_val,
        priority_val_pred,
        labels=[0, 1],
        target_names=["PREFERRED", "MUST"],
        zero_division=0
    )
)


# --------------------------------------------------
# 7. Confusion matrix
# --------------------------------------------------

print("\n=== CONFUSION MATRIX ===")

cm = confusion_matrix(
    y_priority_val,
    priority_val_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual PREFERRED", "Actual MUST"],
    columns=["Predicted PREFERRED", "Predicted MUST"]
)

display(cm_df)

print("\nSTEP 154: COMPLETED")

=== PRIORITY MODEL TRAINING ===
Model: TF-IDF + Logistic Regression
Training requirements: 5328
Validation requirements: 1171
Training completed successfully.

=== PRIORITY VALIDATION RESULTS ===
Accuracy: 0.7011
Macro Precision: 0.6473
Macro Recall: 0.6968
Macro F1: 0.6496

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

   PREFERRED       0.41      0.69      0.52       270
        MUST       0.88      0.70      0.78       901

    accuracy                           0.70      1171
   macro avg       0.65      0.70      0.65      1171
weighted avg       0.77      0.70      0.72      1171


=== CONFUSION MATRIX ===


,Predicted PREFERRED,Predicted MUST
Actual PREFERRED,186,84
Actual MUST,266,635



STEP 154: COMPLETED


In [16]:
# Step 155 - Compare priority model with majority-class baseline

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# DummyClassifier requires an input matrix.
# We use a constant feature because the model ignores input features.

X_dummy_train = pd.DataFrame({
    "constant": [1] * len(y_priority_train)
})

X_dummy_val = pd.DataFrame({
    "constant": [1] * len(y_priority_val)
})

dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_dummy_train,
    y_priority_train
)

dummy_predictions = dummy_model.predict(X_dummy_val)

results = []

for model_name, predictions in [
    ("Majority Class Baseline", dummy_predictions),
    ("TF-IDF + Logistic Regression", priority_val_pred)
]:

    results.append({
        "MODEL": model_name,
        "ACCURACY": accuracy_score(
            y_priority_val, predictions
        ),
        "MACRO_PRECISION": precision_score(
            y_priority_val,
            predictions,
            average="macro",
            zero_division=0
        ),
        "MACRO_RECALL": recall_score(
            y_priority_val,
            predictions,
            average="macro",
            zero_division=0
        ),
        "MACRO_F1": f1_score(
            y_priority_val,
            predictions,
            average="macro",
            zero_division=0
        ),
        "MUST_RECALL": recall_score(
            y_priority_val,
            predictions,
            pos_label=1,
            zero_division=0
        ),
        "PREFERRED_RECALL": recall_score(
            y_priority_val,
            predictions,
            pos_label=0,
            zero_division=0
        )
    })

comparison_df = pd.DataFrame(results)

print("=== PRIORITY MODEL COMPARISON ===")

display(comparison_df.round(4))

print("\nSTEP 155: COMPLETED")

=== PRIORITY MODEL COMPARISON ===


,MODEL,ACCURACY,MACRO_PRECISION,MACRO_RECALL,MACRO_F1,MUST_RECALL,PREFERRED_RECALL
0,Majority Class Baseline,0.7694,0.3847,0.5000,0.4348,1.0000,0.0000
1,TF-IDF + Logistic Regression,0.7011,0.6473,0.6968,0.6496,0.7048,0.6889



STEP 155: COMPLETED


In [17]:
# Step 156 - Priority classification with Linear SVM

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------------
# 1. Build Linear SVM pipeline
# --------------------------------------------------

priority_svm_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=10000,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=42,
            max_iter=5000
        )
    )
])

# --------------------------------------------------
# 2. Train model
# --------------------------------------------------

priority_svm_model.fit(
    X_priority_train,
    y_priority_train
)

# --------------------------------------------------
# 3. Validation predictions
# --------------------------------------------------

priority_svm_val_pred = priority_svm_model.predict(
    X_priority_val
)

# --------------------------------------------------
# 4. Compare models
# --------------------------------------------------

comparison_results = []

models = [
    ("Majority Baseline", dummy_predictions),
    ("Logistic Regression", priority_val_pred),
    ("Linear SVM", priority_svm_val_pred)
]

for model_name, predictions in models:

    comparison_results.append({
        "MODEL": model_name,

        "ACCURACY": accuracy_score(
            y_priority_val, predictions
        ),

        "MACRO_F1": f1_score(
            y_priority_val,
            predictions,
            average="macro",
            zero_division=0
        ),

        "MUST_RECALL": recall_score(
            y_priority_val,
            predictions,
            pos_label=1,
            zero_division=0
        ),

        "PREFERRED_RECALL": recall_score(
            y_priority_val,
            predictions,
            pos_label=0,
            zero_division=0
        )
    })

priority_comparison = pd.DataFrame(comparison_results)

print("=== PRIORITY MODEL COMPARISON ===")

display(priority_comparison.round(4))

# --------------------------------------------------
# 5. Linear SVM detailed results
# --------------------------------------------------

print("\n=== LINEAR SVM CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_priority_val,
        priority_svm_val_pred,
        labels=[0, 1],
        target_names=["PREFERRED", "MUST"],
        zero_division=0
    )
)

print("\n=== LINEAR SVM CONFUSION MATRIX ===")

cm = confusion_matrix(
    y_priority_val,
    priority_svm_val_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual PREFERRED", "Actual MUST"],
    columns=["Predicted PREFERRED", "Predicted MUST"]
)

display(cm_df)

print("\nSTEP 156: COMPLETED")

=== PRIORITY MODEL COMPARISON ===


,MODEL,ACCURACY,MACRO_F1,MUST_RECALL,PREFERRED_RECALL
0,Majority Baseline,0.7694,0.4348,1.0000,0.0000
1,Logistic Regression,0.7011,0.6496,0.7048,0.6889
2,Linear SVM,0.7028,0.6483,0.7125,0.6704



=== LINEAR SVM CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

   PREFERRED       0.41      0.67      0.51       270
        MUST       0.88      0.71      0.79       901

    accuracy                           0.70      1171
   macro avg       0.64      0.69      0.65      1171
weighted avg       0.77      0.70      0.72      1171


=== LINEAR SVM CONFUSION MATRIX ===


,Predicted PREFERRED,Predicted MUST
Actual PREFERRED,181,89
Actual MUST,259,642



STEP 156: COMPLETED


In [18]:
# Step 157 - Requirement-aware feature engineering
# Separate TF-IDF representations for customer text and requirement

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix
)

# --------------------------------------------------
# 1. Prepare separate input columns
# --------------------------------------------------

feature_columns = [
    "LLM_TEXT",
    "REQUIREMENT_DESCRIPTION"
]

X_priority_train_v2 = (
    priority_train[feature_columns]
    .fillna("")
    .copy()
)

X_priority_val_v2 = (
    priority_val[feature_columns]
    .fillna("")
    .copy()
)

# --------------------------------------------------
# 2. Build feature transformer
# --------------------------------------------------

priority_features = ColumnTransformer(
    transformers=[
        (
            "customer_text",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_features=10000,
                sublinear_tf=True
            ),
            "LLM_TEXT"
        ),
        (
            "requirement",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True
            ),
            "REQUIREMENT_DESCRIPTION"
        )
    ]
)

# --------------------------------------------------
# 3. Build model
# --------------------------------------------------

priority_model_v2 = Pipeline([
    ("features", priority_features),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

# --------------------------------------------------
# 4. Train
# --------------------------------------------------

priority_model_v2.fit(
    X_priority_train_v2,
    y_priority_train
)

# --------------------------------------------------
# 5. Validation predictions
# --------------------------------------------------

priority_v2_pred = priority_model_v2.predict(
    X_priority_val_v2
)

# --------------------------------------------------
# 6. Compare results
# --------------------------------------------------

comparison = []

models = [
    ("Logistic Regression V1", priority_val_pred),
    ("Linear SVM", priority_svm_val_pred),
    ("Logistic Regression V2", priority_v2_pred)
]

for name, pred in models:

    comparison.append({
        "MODEL": name,
        "ACCURACY": accuracy_score(y_priority_val, pred),
        "MACRO_F1": f1_score(
            y_priority_val,
            pred,
            average="macro",
            zero_division=0
        ),
        "MUST_RECALL": recall_score(
            y_priority_val,
            pred,
            pos_label=1,
            zero_division=0
        ),
        "PREFERRED_RECALL": recall_score(
            y_priority_val,
            pred,
            pos_label=0,
            zero_division=0
        )
    })

comparison_df_v2 = pd.DataFrame(comparison)

print("=== REQUIREMENT-AWARE MODEL COMPARISON ===")

display(comparison_df_v2.round(4))

print("\n=== V2 CONFUSION MATRIX ===")

cm = confusion_matrix(
    y_priority_val,
    priority_v2_pred,
    labels=[0, 1]
)

display(
    pd.DataFrame(
        cm,
        index=["Actual PREFERRED", "Actual MUST"],
        columns=["Predicted PREFERRED", "Predicted MUST"]
    )
)

print("\nSTEP 157: COMPLETED")

=== REQUIREMENT-AWARE MODEL COMPARISON ===


,MODEL,ACCURACY,MACRO_F1,MUST_RECALL,PREFERRED_RECALL
0,Logistic Regression V1,0.7011,0.6496,0.7048,0.6889
1,Linear SVM,0.7028,0.6483,0.7125,0.6704
2,Logistic Regression V2,0.6909,0.6376,0.6981,0.6667



=== V2 CONFUSION MATRIX ===


,Predicted PREFERRED,Predicted MUST
Actual PREFERRED,180,90
Actual MUST,272,629



STEP 157: COMPLETED


In [19]:
# Step 158 - Requirement context and error analysis

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Prepare validation analysis dataset
# --------------------------------------------------

priority_error_analysis = priority_val[
    [
        "REQUEST_ID",
        "LLM_TEXT",
        "REQUIREMENT_DESCRIPTION",
        "REQUIREMENT_TYPE",
        "PRIORITY_TARGET"
    ]
].copy()

priority_error_analysis["PREDICTED_TARGET"] = priority_val_pred

priority_error_analysis["PREDICTED_PRIORITY"] = (
    priority_error_analysis["PREDICTED_TARGET"]
    .map({1: "MUST", 0: "PREFERRED"})
)

priority_error_analysis["CORRECT"] = (
    priority_error_analysis["PRIORITY_TARGET"]
    == priority_error_analysis["PREDICTED_TARGET"]
)

# --------------------------------------------------
# 2. Identify mixed-priority customer requests
# --------------------------------------------------

priority_counts = (
    priority_val.groupby("REQUEST_ID")["REQUIREMENT_TYPE"]
    .nunique()
)

mixed_request_ids = set(
    priority_counts[priority_counts > 1].index
)

priority_error_analysis["MIXED_PRIORITY_REQUEST"] = (
    priority_error_analysis["REQUEST_ID"]
    .isin(mixed_request_ids)
)

# --------------------------------------------------
# 3. Compare performance by request type
# --------------------------------------------------

summary = (
    priority_error_analysis
    .groupby("MIXED_PRIORITY_REQUEST")
    .agg(
        REQUIREMENTS=("CORRECT", "size"),
        CORRECT_PREDICTIONS=("CORRECT", "sum"),
        ACCURACY=("CORRECT", "mean")
    )
    .reset_index()
)

summary["REQUEST_TYPE"] = summary[
    "MIXED_PRIORITY_REQUEST"
].map({
    False: "Single-priority requests",
    True: "Mixed-priority requests"
})

print("=== PRIORITY CONTEXT ANALYSIS ===")

display(
    summary[
        [
            "REQUEST_TYPE",
            "REQUIREMENTS",
            "CORRECT_PREDICTIONS",
            "ACCURACY"
        ]
    ].round(4)
)

# --------------------------------------------------
# 4. Identify classification errors
# --------------------------------------------------

priority_errors = priority_error_analysis[
    ~priority_error_analysis["CORRECT"]
].copy()

print("\n=== CLASSIFICATION ERRORS ===")

print("Total validation requirements:", len(priority_error_analysis))
print("Total classification errors:", len(priority_errors))

print(
    "Errors in mixed-priority requests:",
    priority_errors["MIXED_PRIORITY_REQUEST"].sum()
)

# --------------------------------------------------
# 5. Error distribution
# --------------------------------------------------

error_distribution = (
    priority_errors
    .groupby(
        [
            "REQUIREMENT_TYPE",
            "PREDICTED_PRIORITY",
            "MIXED_PRIORITY_REQUEST"
        ]
    )
    .size()
    .reset_index(name="ERROR_COUNT")
)

print("\n=== ERROR DISTRIBUTION ===")

display(error_distribution)

# --------------------------------------------------
# 6. Display sample errors with full context
# --------------------------------------------------

print("\n=== SAMPLE MISCLASSIFIED REQUIREMENTS ===")

sample_errors = priority_errors[
    [
        "REQUEST_ID",
        "LLM_TEXT",
        "REQUIREMENT_DESCRIPTION",
        "REQUIREMENT_TYPE",
        "PREDICTED_PRIORITY",
        "MIXED_PRIORITY_REQUEST"
    ]
].head(15)

display(sample_errors)

# --------------------------------------------------
# 7. Save analysis
# --------------------------------------------------

output_path = "../data/interim/priority_error_analysis.csv"

priority_error_analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", output_path)

print("\nSTEP 158: COMPLETED")

=== PRIORITY CONTEXT ANALYSIS ===


,REQUEST_TYPE,REQUIREMENTS,CORRECT_PREDICTIONS,ACCURACY
0,Single-priority requests,370,370,1.000
1,Mixed-priority requests,801,451,0.563



=== CLASSIFICATION ERRORS ===
Total validation requirements: 1171
Total classification errors: 350
Errors in mixed-priority requests: 350

=== ERROR DISTRIBUTION ===


,REQUIREMENT_TYPE,PREDICTED_PRIORITY,MIXED_PRIORITY_REQUEST,ERROR_COUNT
0,MUST,PREFERRED,True,266
1,PREFERRED,MUST,True,84



=== SAMPLE MISCLASSIFIED REQUIREMENTS ===


,REQUEST_ID,LLM_TEXT,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PREDICTED_PRIORITY,MIXED_PRIORITY_REQUEST
19,R0007,I'm looking for an oven that comes shrink-wrap...,CLEANING = CATALYTIC,PREFERRED,MUST,True
21,R0007,I'm looking for an oven that comes shrink-wrap...,KNOB_COLOR = STAINLESS STEEL,MUST,PREFERRED,True
36,R0014,I'm looking for a 45 cm wide steam oven with a...,CLEANING = ETC,MUST,PREFERRED,True
38,R0014,I'm looking for a 45 cm wide steam oven with a...,OVEN_TYPE = STEAM,MUST,PREFERRED,True
39,R0014,I'm looking for a 45 cm wide steam oven with a...,OVEN_VOLUME_L = 82,MUST,PREFERRED,True
40,R0014,I'm looking for a 45 cm wide steam oven with a...,POWER_CABLE = US TYPE PLUG,MUST,PREFERRED,True
41,R0014,I'm looking for a 45 cm wide steam oven with a...,WIDTH_CM = 45,MUST,PREFERRED,True
48,R0016,I'm looking for a white oven with an electroni...,NUMBER_OF_FUNCTIONS = 11,PREFERRED,MUST,True
70,R0024,I'm looking for a convection oven that has eig...,PANEL_COLOR = STAINLESS STEEL,PREFERRED,MUST,True
78,R0028,I'm looking for a 90 cm wide oven with rotary ...,KNOB_STRUCTURE = ROTARY,MUST,PREFERRED,True



Saved: ../data/interim/priority_error_analysis.csv

STEP 158: COMPLETED


In [20]:
# Step 159 - Inspect requirement context in mixed-priority requests

import pandas as pd
from IPython.display import display

# Use the error analysis prepared in Step 158
analysis_df = priority_error_analysis.copy()

# Select mixed-priority requests
mixed_df = analysis_df[
    analysis_df["MIXED_PRIORITY_REQUEST"]
].copy()

# Identify requests containing classification errors
error_request_ids = (
    mixed_df.loc[
        ~mixed_df["CORRECT"],
        "REQUEST_ID"
    ]
    .drop_duplicates()
    .head(15)
    .tolist()
)

print("=== REQUIREMENT CONTEXT ANALYSIS ===")
print("Selected customer requests:", len(error_request_ids))

# Inspect complete requests and their individual requirements
for request_id in error_request_ids:

    request_data = mixed_df[
        mixed_df["REQUEST_ID"] == request_id
    ].copy()

    customer_text = request_data["LLM_TEXT"].iloc[0]

    print("\n" + "=" * 90)
    print("REQUEST ID:", request_id)

    print("\nCUSTOMER TEXT:")
    print(customer_text)

    print("\nREQUIREMENTS:")

    display(
        request_data[
            [
                "REQUIREMENT_DESCRIPTION",
                "REQUIREMENT_TYPE",
                "PREDICTED_PRIORITY",
                "CORRECT"
            ]
        ].reset_index(drop=True)
    )

print("\nSTEP 159: COMPLETED")


=== REQUIREMENT CONTEXT ANALYSIS ===
Selected customer requests: 15

REQUEST ID: R0007

CUSTOMER TEXT:
I'm looking for an oven that comes shrink-wrapped, with stainless steel knobs and an A energy rating; preferably it would have catalytic cleaning and no roasting tray included.

REQUIREMENTS:


,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PREDICTED_PRIORITY,CORRECT
0,CLEANING = CATALYTIC,PREFERRED,MUST,False
1,ENERGY_CLASS = A,MUST,MUST,True
2,KNOB_COLOR = STAINLESS STEEL,MUST,PREFERRED,False
3,PACKAGING_TYPE = SHRINK,MUST,MUST,True
4,ROASTING_TRAY = 0,PREFERRED,PREFERRED,True



REQUEST ID: R0014

CUSTOMER TEXT:
I'm looking for a 45 cm wide steam oven with an 82 L capacity, US plug, and ECO cleaning function, and I'd prefer it to be energy class A++ if possible.

REQUIREMENTS:


,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PREDICTED_PRIORITY,CORRECT
0,CLEANING = ETC,MUST,PREFERRED,False
1,ENERGY_CLASS = A++,PREFERRED,PREFERRED,True
2,OVEN_TYPE = STEAM,MUST,PREFERRED,False
3,OVEN_VOLUME_L = 82,MUST,PREFERRED,False
4,POWER_CABLE = US TYPE PLUG,MUST,PREFERRED,False
5,WIDTH_CM = 45,MUST,PREFERRED,False



REQUEST ID: R0016

CUSTOMER TEXT:
I'm looking for a white oven with an electronic touch timer, a rotisserie spit, two baking trays, and about eleven cooking functions (the number of functions is a preference rather than a strict requirement).

REQUIREMENTS:


,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PREDICTED_PRIORITY,CORRECT
0,BAKING_TRAY = 2,MUST,MUST,True
1,NUMBER_OF_FUNCTIONS = 11,PREFERRED,MUST,False
2,OVEN_CONTROL = ELECTRONIC TOUCH TIMER,MUST,MUST,True
3,PANEL_COLOR = WHITE,MUST,MUST,True
4,TURNSPIT = YES,MUST,MUST,True


... (21 outputs left out)

,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PREDICTED_PRIORITY,CORRECT
0,ENERGY_CLASS = A++,MUST,PREFERRED,False
1,KNOB_TYPE = K1,MUST,PREFERRED,False
2,OVEN_TYPE = STEAM,PREFERRED,MUST,False
3,POWER_CABLE = UK TYPE PLUG,MUST,PREFERRED,False



REQUEST ID: R0225

CUSTOMER TEXT:
I'm looking for a combination oven with A++ energy efficiency and, if possible, a handle-style H2.

REQUIREMENTS:


,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,PREDICTED_PRIORITY,CORRECT
0,ENERGY_CLASS = A++,MUST,MUST,True
1,HANDLE_TYPE = H2,PREFERRED,MUST,False
2,OVEN_TYPE = COMBINATION,MUST,MUST,True



STEP 159: COMPLETED


In [21]:
# Step 160 - Requirement-level context extraction pilot

import re
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 1. Technical terminology aliases
# --------------------------------------------------

# Initial examples only.
# These aliases do not contain priority information.

TERM_ALIASES = {
    "TURNSPIT = YES": [
        "rotisserie",
        "turnspit"
    ],
    "TURNSPIT = NO": [
        "no rotisserie",
        "without rotisserie",
        "no turnspit"
    ],
    "CLEANING = PYROLYSIS": [
        "pyrolysis",
        "pyrolytic"
    ],
    "CLEANING = HYDROCLEAN": [
        "hydroclean",
        "hydro clean"
    ],
    "KNOB_STRUCTURE = ROTARY": [
        "rotary knobs",
        "rotary-style knobs"
    ],
    "OVEN_CONTROL = ELECTRONIC TOUCH TIMER": [
        "electronic touch timer",
        "touch timer"
    ]
}

# --------------------------------------------------
# 2. Split customer text into context segments
# --------------------------------------------------

def split_context_segments(text):

    text = str(text).strip()

    # Split at sentence boundaries, semicolons,
    # and selected coordinating conjunctions.
    segments = re.split(
        r"(?<=[.!?])\s+|;\s*|,\s+and\s+|\s+but\s+",
        text,
        flags=re.IGNORECASE
    )

    return [
        segment.strip(" ,.;")
        for segment in segments
        if segment.strip(" ,.;")
    ]


# --------------------------------------------------
# 3. Build search terms for each requirement
# --------------------------------------------------

def get_requirement_search_terms(requirement):

    requirement = str(requirement).strip()

    terms = TERM_ALIASES.get(requirement, []).copy()

    if " = " in requirement:

        characteristic, value = requirement.split(" = ", 1)

        # Add the technical value as a fallback.
        if value.strip():
            terms.append(value.strip())

    # Longer terms first
    terms = sorted(
        set(terms),
        key=len,
        reverse=True
    )

    return terms


# --------------------------------------------------
# 4. Extract the best matching context
# --------------------------------------------------

def extract_requirement_context(customer_text, requirement):

    segments = split_context_segments(customer_text)

    search_terms = get_requirement_search_terms(requirement)

    matches = []

    for segment in segments:

        segment_lower = segment.lower()

        matched_terms = [
            term
            for term in search_terms
            if term.lower() in segment_lower
        ]

        if matched_terms:

            matches.append({
                "SEGMENT": segment,
                "MATCHED_TERMS": matched_terms,
                "SCORE": max(len(term) for term in matched_terms)
            })

    if not matches:
        return {
            "EXTRACTED_CONTEXT": None,
            "MATCH_STATUS": "NOT_FOUND",
            "MATCHED_TERM": None
        }

    best_match = max(
        matches,
        key=lambda item: item["SCORE"]
    )

    return {
        "EXTRACTED_CONTEXT": best_match["SEGMENT"],
        "MATCH_STATUS": "MATCHED",
        "MATCHED_TERM": best_match["MATCHED_TERMS"][0]
    }


# --------------------------------------------------
# 5. Apply to the Step 159 pilot requests
# --------------------------------------------------

pilot_request_ids = error_request_ids

context_pilot = priority_error_analysis[
    priority_error_analysis["REQUEST_ID"].isin(
        pilot_request_ids
    )
].copy()

context_results = context_pilot.apply(
    lambda row: extract_requirement_context(
        row["LLM_TEXT"],
        row["REQUIREMENT_DESCRIPTION"]
    ),
    axis=1
)

context_results = pd.DataFrame(
    context_results.tolist(),
    index=context_pilot.index
)

context_pilot = pd.concat(
    [context_pilot, context_results],
    axis=1
)

# --------------------------------------------------
# 6. Summarize extraction coverage
# --------------------------------------------------

print("=== REQUIREMENT CONTEXT EXTRACTION PILOT ===")

print("Requests:", context_pilot["REQUEST_ID"].nunique())
print("Requirements:", len(context_pilot))

print("\n=== MATCH STATUS ===")

display(
    context_pilot["MATCH_STATUS"]
    .value_counts()
    .rename_axis("MATCH_STATUS")
    .reset_index(name="COUNT")
)

match_rate = (
    context_pilot["MATCH_STATUS"].eq("MATCHED").mean()
)

print(f"\nContext match rate: {match_rate:.2%}")

# --------------------------------------------------
# 7. Inspect the extracted contexts
# --------------------------------------------------

print("\n=== CONTEXT EXTRACTION RESULTS ===")

display(
    context_pilot[
        [
            "REQUEST_ID",
            "REQUIREMENT_DESCRIPTION",
            "REQUIREMENT_TYPE",
            "EXTRACTED_CONTEXT",
            "MATCH_STATUS",
            "MATCHED_TERM"
        ]
    ].head(30)
)

print("\nSTEP 160: COMPLETED")

=== REQUIREMENT CONTEXT EXTRACTION PILOT ===
Requests: 15
Requirements: 59

=== MATCH STATUS ===


,MATCH_STATUS,COUNT
0,MATCHED,42
1,NOT_FOUND,17



Context match rate: 71.19%

=== CONTEXT EXTRACTION RESULTS ===


,REQUEST_ID,REQUIREMENT_DESCRIPTION,REQUIREMENT_TYPE,EXTRACTED_CONTEXT,MATCH_STATUS,MATCHED_TERM
19,R0007,CLEANING = CATALYTIC,PREFERRED,preferably it would have catalytic cleaning an...,MATCHED,CATALYTIC
20,R0007,ENERGY_CLASS = A,MUST,I'm looking for an oven that comes shrink-wrap...,MATCHED,A
21,R0007,KNOB_COLOR = STAINLESS STEEL,MUST,I'm looking for an oven that comes shrink-wrap...,MATCHED,STAINLESS STEEL
22,R0007,PACKAGING_TYPE = SHRINK,MUST,I'm looking for an oven that comes shrink-wrap...,MATCHED,SHRINK
23,R0007,ROASTING_TRAY = 0,PREFERRED,None,NOT_FOUND,None
36,R0014,CLEANING = ETC,MUST,None,NOT_FOUND,None
37,R0014,ENERGY_CLASS = A++,PREFERRED,I'd prefer it to be energy class A++ if possible,MATCHED,A++
38,R0014,OVEN_TYPE = STEAM,MUST,I'm looking for a 45 cm wide steam oven with a...,MATCHED,STEAM
39,R0014,OVEN_VOLUME_L = 82,MUST,I'm looking for a 45 cm wide steam oven with a...,MATCHED,82
40,R0014,POWER_CABLE = US TYPE PLUG,MUST,None,NOT_FOUND,None



STEP 160: COMPLETED


In [22]:
# Step 161 - Context extraction quality diagnostics

import pandas as pd

diagnostics = context_pilot.copy()

# --------------------------------------------------
# 1. Missing context matches
# --------------------------------------------------

missing_context = diagnostics[
    diagnostics["MATCH_STATUS"] == "NOT_FOUND"
].copy()

print("=== MISSING CONTEXT MATCHES ===")
print("Total missing:", len(missing_context))

display(
    missing_context[
        [
            "REQUEST_ID",
            "REQUIREMENT_DESCRIPTION",
            "LLM_TEXT"
        ]
    ].reset_index(drop=True)
)

# --------------------------------------------------
# 2. Detect shared context segments
# --------------------------------------------------

matched_context = diagnostics[
    diagnostics["MATCH_STATUS"] == "MATCHED"
].copy()

context_usage = (
    matched_context
    .groupby(
        ["REQUEST_ID", "EXTRACTED_CONTEXT"],
        dropna=False
    )
    .agg(
        REQUIREMENT_COUNT=(
            "REQUIREMENT_DESCRIPTION",
            "size"
        ),
        REQUIREMENTS=(
            "REQUIREMENT_DESCRIPTION",
            lambda values: list(values)
        )
    )
    .reset_index()
)

shared_context = context_usage[
    context_usage["REQUIREMENT_COUNT"] > 1
].copy()

print("\n=== SHARED CONTEXT SEGMENTS ===")
print("Shared segments:", len(shared_context))

display(shared_context)

# --------------------------------------------------
# 3. Inspect suspicious short matching terms
# --------------------------------------------------

diagnostics["MATCHED_TERM_LENGTH"] = (
    diagnostics["MATCHED_TERM"]
    .fillna("")
    .astype(str)
    .str.len()
)

short_matches = diagnostics[
    (diagnostics["MATCH_STATUS"] == "MATCHED")
    & (diagnostics["MATCHED_TERM_LENGTH"] <= 2)
].copy()

print("\n=== SHORT MATCHING TERMS ===")
print("Potentially ambiguous matches:", len(short_matches))

display(
    short_matches[
        [
            "REQUEST_ID",
            "REQUIREMENT_DESCRIPTION",
            "MATCHED_TERM",
            "EXTRACTED_CONTEXT"
        ]
    ]
)

# --------------------------------------------------
# 4. Summary
# --------------------------------------------------

print("\n=== CONTEXT DIAGNOSTIC SUMMARY ===")

print("Total requirements:", len(diagnostics))
print("Matched:", len(matched_context))
print("Not found:", len(missing_context))
print("Shared context segments:", len(shared_context))
print("Short matching terms:", len(short_matches))

print("\nSTEP 161: COMPLETED")

=== MISSING CONTEXT MATCHES ===
Total missing: 17


,REQUEST_ID,REQUIREMENT_DESCRIPTION,LLM_TEXT
0,R0007,ROASTING_TRAY = 0,I'm looking for an oven that comes shrink-wrap...
1,R0014,CLEANING = ETC,I'm looking for a 45 cm wide steam oven with a...
2,R0014,POWER_CABLE = US TYPE PLUG,I'm looking for a 45 cm wide steam oven with a...
3,R0016,BAKING_TRAY = 2,I'm looking for a white oven with an electroni...
4,R0016,NUMBER_OF_FUNCTIONS = 11,I'm looking for a white oven with an electroni...
5,R0024,NUMBER_OF_FUNCTIONS = 8,I'm looking for a convection oven that has eig...
6,R0024,PANEL_COLOR = STAINLESS STEEL,I'm looking for a convection oven that has eig...
7,R0028,WIFI = YES,I'm looking for a 90 cm wide oven with rotary ...
8,R0037,TELESCOPIC_RAILS = 2,I'm looking for a 50-liter oven with two teles...
9,R0068,WIRE_SHELF = 2,I'm looking for a combination oven with stainl...



=== SHARED CONTEXT SEGMENTS ===
Shared segments: 15


,REQUEST_ID,EXTRACTED_CONTEXT,REQUIREMENT_COUNT,REQUIREMENTS
0,R0007,I'm looking for an oven that comes shrink-wrap...,3,"[ENERGY_CLASS = A, KNOB_COLOR = STAINLESS STEE..."
3,R0014,I'm looking for a 45 cm wide steam oven with a...,3,"[OVEN_TYPE = STEAM, OVEN_VOLUME_L = 82, WIDTH_..."
4,R0016,I'm looking for a white oven with an electroni...,3,"[OVEN_CONTROL = ELECTRONIC TOUCH TIMER, PANEL_..."
5,R0024,I'm looking for a convection oven that has eig...,2,"[CLEANING = ETC, OVEN_TYPE = CONVECTION]"
6,R0028,I'm looking for a 90 cm wide oven with rotary ...,2,"[KNOB_STRUCTURE = ROTARY, WIDTH_CM = 90]"
8,R0036,I'm looking for a silver-handled oven (no pizz...,3,"[CLEANING = HYDROCLEAN, HANDLE_COLOR = SILVER,..."
11,R0068,I'm looking for a combination oven with stainl...,2,"[KNOB_COLOR = STAINLESS STEEL, OVEN_TYPE = COM..."
12,R0077,I'm looking for a dark silver oven with an ele...,2,"[OVEN_CONTROL = ELECTRONIC TOUCH TIMER, PANEL_..."
13,R0077,ideally it would be a 45 cm wide combination oven,2,"[OVEN_TYPE = COMBINATION, WIDTH_CM = 45]"
14,R0129,I'm looking for a built-in oven with a silver ...,2,"[ENERGY_CLASS = A, HANDLE_COLOR = SILVER]"



=== SHORT MATCHING TERMS ===
Potentially ambiguous matches: 13


,REQUEST_ID,REQUIREMENT_DESCRIPTION,MATCHED_TERM,EXTRACTED_CONTEXT
20,R0007,ENERGY_CLASS = A,A,I'm looking for an oven that comes shrink-wrap...
39,R0014,OVEN_VOLUME_L = 82,82,I'm looking for a 45 cm wide steam oven with a...
41,R0014,WIDTH_CM = 45,45,I'm looking for a 45 cm wide steam oven with a...
80,R0028,WIDTH_CM = 90,90,I'm looking for a 90 cm wide oven with rotary ...
106,R0036,PIZZA_TRAY = NO,NO,I'm looking for a silver-handled oven (no pizz...
107,R0037,OVEN_VOLUME_L = 50,50,I'm looking for a 50-liter oven with two teles...
192,R0077,WIDTH_CM = 45,45,ideally it would be a 45 cm wide combination oven
320,R0129,ENERGY_CLASS = A,A,I'm looking for a built-in oven with a silver ...
323,R0131,ENERGY_CLASS = A,A,I'm looking for a stainless steel oven with en...
402,R0158,WIDTH_CM = 45,45,I'm looking for a 45 cm wide oven with an elec...



=== CONTEXT DIAGNOSTIC SUMMARY ===
Total requirements: 59
Matched: 42
Not found: 17
Shared context segments: 15
Short matching terms: 13

STEP 161: COMPLETED


In [23]:
# Step 162 - Save NLP model evaluation summary

import pandas as pd
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

model_summary = pd.DataFrame([
    {
        "TASK": "Requirement Extraction",
        "MODEL": "TF-IDF + Logistic Regression",
        "DATASET": "Validation",
        "MICRO_F1": 0.8748,
        "MACRO_F1": 0.8855,
        "EXACT_MATCH": 0.3906,
        "STATUS": "Baseline"
    },
    {
        "TASK": "Requirement Extraction",
        "MODEL": "TF-IDF + Linear SVM",
        "DATASET": "Test",
        "MICRO_F1": 0.9299,
        "MACRO_F1": 0.9130,
        "EXACT_MATCH": 0.6477,
        "STATUS": "Selected"
    },
    {
        "TASK": "Priority Classification",
        "MODEL": "TF-IDF + Logistic Regression",
        "DATASET": "Validation",
        "MICRO_F1": None,
        "MACRO_F1": 0.6496,
        "EXACT_MATCH": None,
        "STATUS": "Selected"
    },
    {
        "TASK": "Priority Classification",
        "MODEL": "TF-IDF + Linear SVM",
        "DATASET": "Validation",
        "MICRO_F1": None,
        "MACRO_F1": 0.6483,
        "EXACT_MATCH": None,
        "STATUS": "Alternative"
    }
])

output_path = output_dir / "nlp_model_summary.csv"

model_summary.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=== NLP MODEL SUMMARY ===")
display(model_summary)

print("\nSaved:", output_path)
print("\nSTEP 162: COMPLETED")

=== NLP MODEL SUMMARY ===


,TASK,MODEL,DATASET,MICRO_F1,MACRO_F1,EXACT_MATCH,STATUS
0,Requirement Extraction,TF-IDF + Logistic Regression,Validation,0.8748,0.8855,0.3906,Baseline
1,Requirement Extraction,TF-IDF + Linear SVM,Test,0.9299,0.9130,0.6477,Selected
2,Priority Classification,TF-IDF + Logistic Regression,Validation,NaN,0.6496,NaN,Selected
3,Priority Classification,TF-IDF + Linear SVM,Validation,NaN,0.6483,NaN,Alternative



Saved: ../data/processed/nlp_model_summary.csv

STEP 162: COMPLETED


## Step 180 — Save Selected NLP Models

Save the trained NLP models and their preprocessing components
for integration with the Product Matching module.

Artifacts:
- NLP 1: TF-IDF vectorizer, Linear SVM, label mapping, threshold.
- NLP 2: TF-IDF + Logistic Regression pipeline.

The saved models will be loaded in Notebook 05.

In [24]:
# STEP 180 — Save Selected NLP Models

from pathlib import Path
import joblib

# Output directory
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Verify trained models are available
assert hasattr(tfidf_vectorizer, "vocabulary_")
assert hasattr(svm_model, "estimators_")
assert hasattr(priority_model, "classes_")

# NLP 1 — Requirement Extraction
nlp1_artifact = {
    "vectorizer": tfidf_vectorizer,
    "model": svm_model,
    "labels": y_train.columns.tolist(),
    "threshold": final_threshold
}

nlp1_path = MODEL_DIR / "nlp1_requirement_extraction.joblib"

joblib.dump(nlp1_artifact, nlp1_path)

# NLP 2 — Priority Classification
nlp2_artifact = {
    "pipeline": priority_model,
    "class_mapping": {
        0: "PREFERRED",
        1: "MUST"
    }
}

nlp2_path = MODEL_DIR / "nlp2_priority_classification.joblib"

joblib.dump(nlp2_artifact, nlp2_path)

print("=== SAVED NLP MODELS ===")
print("NLP 1:", nlp1_path)
print("NLP 2:", nlp2_path)

print("\n=== NLP 1 ===")
print("Labels:", len(nlp1_artifact["labels"]))
print("Threshold:", nlp1_artifact["threshold"])

print("\n=== NLP 2 ===")
print("Classes:", priority_model.classes_)

print("\nSTEP 180: COMPLETED")

=== SAVED NLP MODELS ===
NLP 1: ../models/nlp1_requirement_extraction.joblib
NLP 2: ../models/nlp2_priority_classification.joblib

=== NLP 1 ===
Labels: 99
Threshold: -0.3

=== NLP 2 ===
Classes: [0 1]

STEP 180: COMPLETED


## Step 188B — Export Original NLP Test Request IDs

Export the original test request IDs used by both NLP models.

This ensures that the end-to-end Product Matching evaluation
uses the same held-out customer requests as the NLP evaluation.

No new train/validation/test split is performed.

In [25]:
# STEP 188B — Export Original NLP Test Request IDs

import pandas as pd
from pathlib import Path

output_path = Path("../data/processed/nlp_test_request_ids.csv")

# Verify original test IDs exist
assert "test_ids" in globals(), (
    "Original test_ids not found. "
    "Run the data split step in Notebook 04 first."
)

# Export original test IDs
test_ids_export = pd.DataFrame({
    "REQUEST_ID": sorted(test_ids)
})

# Integrity checks
assert test_ids_export["REQUEST_ID"].is_unique
assert test_ids_export["REQUEST_ID"].notna().all()

test_ids_export.to_csv(
    output_path,
    index=False
)

print("=== ORIGINAL NLP TEST REQUEST IDS ===")
print("Test requests:", len(test_ids_export))
print("Unique IDs:", test_ids_export["REQUEST_ID"].nunique())

print("\n=== SAMPLE TEST IDS ===")
display(test_ids_export.head(10))

print("\nSaved:", output_path)

print("\nSTEP 188B: COMPLETED")

=== ORIGINAL NLP TEST REQUEST IDS ===
Test requests: 298
Unique IDs: 298

=== SAMPLE TEST IDS ===


,REQUEST_ID
0,R0002
1,R0003
2,R0009
3,R0015
4,R0018
5,R0025
6,R0030
7,R0038
8,R0054
9,R0062



Saved: ../data/processed/nlp_test_request_ids.csv

STEP 188B: COMPLETED
